# ECG (PTB-XL) — train from scratch (Google Colab)

Colab port of [`notebooks/kaggle/01_ecg_ptbxl_training.ipynb`](../kaggle/01_ecg_ptbxl_training.ipynb).
Same model, same training/eval code — only the **setup, dataset acquisition,
and storage paths** differ from the Kaggle version, since Colab has no
"Add Data" mount and wipes `/content` when the runtime disconnects.

See `notebooks/kaggle/01_ecg_ptbxl_training.ipynb` for the full reasoning
behind every choice below (official folds, weighted BCE, tracking MI
sensitivity separately, etc.) — this notebook keeps the same explanations
inline but won't repeat all of them.

## Colab setup (do this first)
1. **Runtime → Change runtime type → T4 GPU** (free tier). Colab free sessions
   disconnect after ~90 min idle and cap out around 12h — this training run
   (a few hours on PTB-XL) fits comfortably; if you get disconnected mid-run,
   Drive-mounted checkpoints (below) mean you don't lose everything.
2. **Mount Google Drive** (cell 2) so the checkpoint and the downloaded
   dataset survive a disconnect — `/content` itself is ephemeral.
3. PTB-XL is **fully open** (no Kaggle account or DUA needed) — we download it
   directly from PhysioNet in cell 3.

In [ ]:
# Cell 1 — install packages Colab's base image doesn't ship with.
!pip install -q wfdb neurokit2

In [ ]:
# Cell 2 — mount Google Drive for persistent storage across disconnects.
from google.colab import drive
drive.mount("/content/drive")

import os
DRIVE_WORKDIR = "/content/drive/MyDrive/aegis-dx/ecg-ptbxl"
os.makedirs(DRIVE_WORKDIR, exist_ok=True)
print("Persistent working dir:", DRIVE_WORKDIR)

## Download PTB-XL from PhysioNet

PTB-XL is fully open (a quick click-through registration on PhysioNet, no
Kaggle account needed). We download once into Drive so re-running this
notebook later doesn't re-download ~3GB every time.

In [ ]:
# Cell 3 — download PTB-XL (skips re-download if already present in Drive).
PTBXL_ROOT = os.path.join(DRIVE_WORKDIR, "ptb-xl-1.0.3")

if not os.path.exists(os.path.join(PTBXL_ROOT, "ptbxl_database.csv")):
    !wget -q -r -N -c -np -P /content/ptbxl_download https://physionet.org/files/ptb-xl/1.0.3/
    downloaded_root = "/content/ptbxl_download/physionet.org/files/ptb-xl/1.0.3"
    !cp -r {downloaded_root} {PTBXL_ROOT}
else:
    print("Already downloaded to Drive, skipping.")

print("PTB-XL root:", PTBXL_ROOT)
assert os.path.exists(os.path.join(PTBXL_ROOT, "ptbxl_database.csv")), "Download failed - check the wget output above."

In [ ]:
# Cell 4 — imports and reproducibility (same as the Kaggle version).
import json
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import wfdb
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type != "cuda":
    print("WARNING: no GPU detected - check Runtime > Change runtime type > T4 GPU.")

## Build multi-label superclass targets

Identical logic to the Kaggle notebook: map `scp_codes` to the 5 PTB-XL
diagnostic superclasses (`NORM`, `MI`, `STTC`, `CD`, `HYP`).

In [ ]:
# Cell 5 — load metadata and the SCP statement -> superclass mapping.
SUPERCLASSES = ["NORM", "MI", "STTC", "CD", "HYP"]

ptbxl_df = pd.read_csv(os.path.join(PTBXL_ROOT, "ptbxl_database.csv"), index_col="ecg_id")
ptbxl_df.scp_codes = ptbxl_df.scp_codes.apply(lambda x: eval(x))

scp_df = pd.read_csv(os.path.join(PTBXL_ROOT, "scp_statements.csv"), index_col=0)
scp_df = scp_df[scp_df.diagnostic == 1]


def scp_codes_to_superclasses(scp_codes: dict) -> list[str]:
    labels = set()
    for code in scp_codes.keys():
        if code in scp_df.index:
            labels.add(scp_df.loc[code].diagnostic_class)
    return sorted(labels & set(SUPERCLASSES))


ptbxl_df["superclasses"] = ptbxl_df.scp_codes.apply(scp_codes_to_superclasses)
ptbxl_df = ptbxl_df[ptbxl_df.superclasses.apply(len) > 0]

for superclass in SUPERCLASSES:
    ptbxl_df[superclass] = ptbxl_df.superclasses.apply(lambda labels: int(superclass in labels))

print(ptbxl_df[SUPERCLASSES].sum())

In [ ]:
# Cell 6 — official train/val/test split by strat_fold (never re-split this).
train_df = ptbxl_df[ptbxl_df.strat_fold <= 8]
val_df = ptbxl_df[ptbxl_df.strat_fold == 9]
test_df = ptbxl_df[ptbxl_df.strat_fold == 10]
print(f"train={len(train_df)} val={len(val_df)} test={len(test_df)}")

In [ ]:
# Cell 7 — Dataset class (100Hz release, per-lead z-normalized). Identical to Kaggle.
class PTBXLDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, root: str):
        self.dataframe = dataframe.reset_index()
        self.root = root

    def __len__(self) -> int:
        return len(self.dataframe)

    def __getitem__(self, index: int):
        row = self.dataframe.iloc[index]
        record_path = os.path.join(self.root, row.filename_lr)
        signal, _ = wfdb.rdsamp(record_path)
        signal = signal.T.astype("float32")

        mean = signal.mean(axis=1, keepdims=True)
        std = signal.std(axis=1, keepdims=True) + 1e-6
        signal = (signal - mean) / std

        labels = row[SUPERCLASSES].values.astype("float32")
        return torch.from_numpy(signal), torch.from_numpy(labels)


train_ds = PTBXLDataset(train_df, PTBXL_ROOT)
val_ds = PTBXLDataset(val_df, PTBXL_ROOT)
test_ds = PTBXLDataset(test_df, PTBXL_ROOT)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=2, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False, num_workers=2)

In [ ]:
# Cell 8 — model: same 1D-ResNet as the Kaggle version.
class ResidualBlock1D(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, stride: int = 1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=7, stride=stride, padding=3, bias=False)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=7, padding=3, bias=False)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = None
        if stride != 1 or in_channels != out_channels:
            self.downsample = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(out_channels),
            )

    def forward(self, x):
        identity = x if self.downsample is None else self.downsample(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(out + identity)


class ECGResNet1D(nn.Module):
    def __init__(self, in_leads: int = 12, num_classes: int = len(SUPERCLASSES)):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(in_leads, 32, kernel_size=15, stride=2, padding=7, bias=False),
            nn.BatchNorm1d(32),
            nn.ReLU(inplace=True),
        )
        self.layer1 = ResidualBlock1D(32, 64, stride=2)
        self.layer2 = ResidualBlock1D(64, 128, stride=2)
        self.layer3 = ResidualBlock1D(128, 256, stride=2)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.pool(x).squeeze(-1)
        return self.head(x)


model = ECGResNet1D().to(DEVICE)
print(sum(p.numel() for p in model.parameters()) / 1e6, "M parameters")

In [ ]:
# Cell 9 — weighted multi-label BCE loss (same imbalance handling as Kaggle).
positive_counts = train_df[SUPERCLASSES].sum().values
negative_counts = len(train_df) - positive_counts
pos_weight = torch.tensor(negative_counts / np.clip(positive_counts, 1, None), dtype=torch.float32).to(DEVICE)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

## Training loop with Drive checkpointing

**The one real Colab-specific change that matters:** free-tier Colab can
disconnect you mid-run. We save the best checkpoint to **Drive** (not
`/content`) after every epoch that improves, so a disconnect loses at most
the current epoch, not the whole run.

In [ ]:
# Cell 10 — training + validation loop, checkpointing to Drive each improvement.
def macro_auroc(y_true: np.ndarray, y_score: np.ndarray) -> float:
    scores = []
    for class_index in range(y_true.shape[1]):
        if len(np.unique(y_true[:, class_index])) < 2:
            continue
        scores.append(roc_auc_score(y_true[:, class_index], y_score[:, class_index]))
    return float(np.mean(scores))


@torch.no_grad()
def evaluate(loader: DataLoader) -> tuple[float, np.ndarray, np.ndarray]:
    model.eval()
    all_true, all_score = [], []
    for signals, labels in loader:
        signals = signals.to(DEVICE)
        logits = model(signals)
        all_true.append(labels.numpy())
        all_score.append(torch.sigmoid(logits).cpu().numpy())
    y_true = np.concatenate(all_true)
    y_score = np.concatenate(all_score)
    return macro_auroc(y_true, y_score), y_true, y_score


CHECKPOINT_PATH = os.path.join(DRIVE_WORKDIR, "ecg_ptbxl_resnet1d_best.pt")
NUM_EPOCHS = 20
best_val_auroc = 0.0

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    for signals, labels in train_loader:
        signals, labels = signals.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(signals)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * signals.size(0)
    scheduler.step()

    train_loss = running_loss / len(train_ds)
    val_auroc, _, _ = evaluate(val_loader)
    print(f"epoch {epoch+1:02d}  train_loss={train_loss:.4f}  val_macro_auroc={val_auroc:.4f}")

    if val_auroc > best_val_auroc:
        best_val_auroc = val_auroc
        torch.save(model.state_dict(), CHECKPOINT_PATH)  # write straight to Drive

model.load_state_dict(torch.load(CHECKPOINT_PATH))
print("Best val macro-AUROC:", best_val_auroc)

In [ ]:
# Cell 11 — held-out test evaluation, same metrics as the Kaggle version.
test_auroc, y_true, y_score = evaluate(test_loader)
print("Test macro-AUROC:", test_auroc)

for class_index, superclass in enumerate(SUPERCLASSES):
    class_auroc = roc_auc_score(y_true[:, class_index], y_score[:, class_index])
    print(f"  {superclass}: AUROC={class_auroc:.4f}")

mi_index = SUPERCLASSES.index("MI")
mi_predictions = (y_score[:, mi_index] >= 0.5).astype(int)
mi_true_positive = ((mi_predictions == 1) & (y_true[:, mi_index] == 1)).sum()
mi_actual_positive = (y_true[:, mi_index] == 1).sum()
mi_sensitivity = mi_true_positive / max(mi_actual_positive, 1)
print(f"MI sensitivity @0.5 threshold: {mi_sensitivity:.4f}  ({mi_true_positive}/{mi_actual_positive})")

In [ ]:
# Cell 12 — the checkpoint is already in Drive from cell 10; write the metadata sidecar alongside it.
metadata = {
    "model": "ECGResNet1D",
    "superclasses": SUPERCLASSES,
    "sampling_rate_hz": 100,
    "input_shape": [12, 1000],
    "test_macro_auroc": test_auroc,
    "mi_sensitivity_at_0.5": float(mi_sensitivity),
    "trained_on": "PTB-XL, official strat_fold split (train 1-8, val 9, test 10)",
    "trained_on_platform": "Google Colab",
}
with open(os.path.join(DRIVE_WORKDIR, "ecg_ptbxl_resnet1d_best.json"), "w") as handle:
    json.dump(metadata, handle, indent=2)

print("Checkpoint + metadata saved under:", DRIVE_WORKDIR)
print(json.dumps(metadata, indent=2))

## Next steps

Same as the Kaggle version — subgroup breakdown, calibration, registry entry,
then serve behind a `SpecialistPort` adapter. See
`notebooks/kaggle/01_ecg_ptbxl_training.ipynb`'s final cell for the full list.